In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split
import pickle
from sklearn.metrics import accuracy_score, roc_auc_score
from ucimlrepo import fetch_ucirepo 



In [ ]:
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144) 
  
# data (as pandas dataframes) 
X = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 

In [13]:
df_credit = pd.concat([X, y], axis=1)
df_credit.columns=["status", "duration", "credit_history", "purpose", "amount", 
                "savings", "employment_duration", "installment_rate",
                "sex", "other_debtors",
                "present_residence", "property",
                "age", "other_installment_plans",
                "housing", "number_credits",
                "job", "people_liable", "telephone", "foreign_worker",
                "target_credit"]
df_credit.dropna(inplace=True)

# Remap target_credit: 1 (good) -> 0, 2 (bad) -> 1
df_credit['target_credit'] = df_credit['target_credit'].map({1: 0, 2: 1})

# Convert categorical string codes to integers
# Status: A11=1, A12=2, A13=3, A14=4
# Status (checking account): 1 = < 0 DM, 2 = 0-200 DM, 3 = >= 200 DM / salary account, 4 = no checking account
status_map = {'A11': 1, 'A12': 2, 'A13': 3, 'A14': 4}
# Credit history: A30=1, A31=2, A32=3, A33=4, A34=5
credit_history_map = {'A30': 1, 'A31': 2, 'A32': 3, 'A33': 4, 'A34': 5}
# Purpose: A40=1, A41=2, A42=3, A43=4, A44=5, A45=6, A46=7, A48=8, A49=9, A410=10
purpose_map = {'A40': 1, 'A41': 2, 'A42': 3, 'A43': 4, 'A44': 5, 'A45': 6, 'A46': 7, 'A48': 8, 'A49': 9, 'A410': 10}
# Savings: A61=1, A62=2, A63=3, A64=4, A65=5
# Savings: 1 = < 100 DM, 2 = 100-500 DM, 3 = 500-1000 DM, 4 = >= 1000 DM, 5 = unknown / no savings account
savings_map = {'A61': 1, 'A62': 2, 'A63': 3, 'A64': 4, 'A65': 5}
# Employment duration: A71=1, A72=2, A73=3, A74=4, A75=5
employment_map = {'A71': 1, 'A72': 2, 'A73': 3, 'A74': 4, 'A75': 5}
# Personal status/sex: A91=male, A92=female, A93=male, A94=male, A95=female
personal_status_map = {'A91': 0, 'A92': 1, 'A93': 0, 'A94': 0, 'A95': 1}
# Other debtors: A101=1, A102=2, A103=3
other_debtors_map = {'A101': 1, 'A102': 2, 'A103': 3}
# Property: A121=1, A122=2, A123=3, A124=4
property_map = {'A121': 1, 'A122': 2, 'A123': 3, 'A124': 4}
# Other installment plans: A141=1, A142=2, A143=3
installment_plans_map = {'A141': 1, 'A142': 2, 'A143': 3}
# Housing: A151=1, A152=2, A153=3
# Housing: 1 = rent, 2 = own, 3 = for free
housing_map = {'A151': 1, 'A152': 2, 'A153': 3}
# Job: A171=1, A172=2, A173=3, A174=4
job_map = {'A171': 1, 'A172': 2, 'A173': 3, 'A174': 4}
# Telephone: A191=1, A192=2
telephone_map = {'A191': 1, 'A192': 2}
# Foreign worker: A201=1, A202=2
foreign_worker_map = {'A201': 1, 'A202': 2}

# Apply mappings
df_credit['status'] = df_credit['status'].map(status_map)
df_credit['credit_history'] = df_credit['credit_history'].map(credit_history_map)
df_credit['purpose'] = df_credit['purpose'].map(purpose_map)
df_credit['savings'] = df_credit['savings'].map(savings_map)
df_credit['employment_duration'] = df_credit['employment_duration'].map(employment_map)
df_credit['sex'] = df_credit['sex'].map(personal_status_map)
df_credit['other_debtors'] = df_credit['other_debtors'].map(other_debtors_map)
df_credit['property'] = df_credit['property'].map(property_map)
df_credit['other_installment_plans'] = df_credit['other_installment_plans'].map(installment_plans_map)
df_credit['housing'] = df_credit['housing'].map(housing_map)
df_credit['job'] = df_credit['job'].map(job_map)
df_credit['telephone'] = df_credit['telephone'].map(telephone_map)
df_credit['foreign_worker'] = df_credit['foreign_worker'].map(foreign_worker_map)

# Ensure all values are numeric
df_credit = df_credit.astype(int)

In [14]:
df_credit.shape
#returns (1000, 21): 1000 instances with 20 features, plus one target

(1000, 21)

In [15]:
# Protected attributes: excluded from the model entirely
protected_attributes = ["age", "sex", "foreign_worker"]

# Actionable features: the only features a user can select in PerCoF
actionable_features = [
    # numeric
    "duration", "amount", "installment_rate", "number_credits",
    # ordinal
    "savings", "status", "job",
    # nominal
    "housing", "other_debtors", "property", "other_installment_plans",
]

# All other model features (credit_history, present_residence,
# employment_duration, purpose, people_liable, telephone) are fixed at the applicant's values

In [16]:
# Feature types (used by PerCoF to decide how each feature may change)
numerical_features = ["duration", "amount", "installment_rate", "present_residence",
                      "number_credits", "people_liable", "age"]
ordinal_features   = ["status", "credit_history", "savings", "employment_duration", "job", "telephone"]
nominal_features   = ["purpose", "other_debtors", "property", "other_installment_plans",
                      "housing", "sex", "foreign_worker"]
categorical_features = ordinal_features + nominal_features

In [17]:
random_state = 1234
credit_train, credit_test = train_test_split(
    df_credit, test_size=0.2, random_state=random_state,
    stratify=df_credit["target_credit"]
)

credit_train.to_parquet("train_cleaned.parquet")
credit_test.to_parquet("test_cleaned.parquet")

# Separate features and target (keep all features including protected attributes)
x_train = credit_train.drop(columns=["target_credit"])
y_train = credit_train["target_credit"]
x_test = credit_test.drop(columns=["target_credit"])
y_test = credit_test["target_credit"]

In [18]:
# random forest model WITHOUT protected attributes 

# Create training and test sets without protected attributes for the model
x_train_model = x_train.drop(columns=protected_attributes)
x_test_model = x_test.drop(columns=protected_attributes)

rf_model = RandomForestClassifier(random_state=random_state, class_weight='balanced')
rf_model.fit(x_train_model, y_train)

# evaluate the model on accuracy and AUC metrics
rf_pred = rf_model.predict(x_test_model)
rf_pred_proba = rf_model.predict_proba(x_test_model)[:, 1]  # Get probability for class 1
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"Accuracy: {rf_accuracy * 100:.2f}%")
rf_auc = roc_auc_score(y_test, rf_pred_proba)
print(f"AUC: {rf_auc * 100:.2f}%")

# Save the model
with open('RF.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

Accuracy: 79.00%
AUC: 84.14%


In [19]:
FEATURE_SPEC = {
    # numeric
    "duration":          {"type": "numeric", "integer": True},
    "amount":            {"type": "numeric", "integer": True},
    "installment_rate":  {"type": "numeric", "integer": True},
    "number_credits":    {"type": "numeric", "integer": True},
    "people_liable":     {"type": "numeric", "integer": True},
    "present_residence": {"type": "numeric", "integer": True},
    # ordinal (order = from worst to best)
    "status":   {"type": "ordinal", "order": [1, 2, 3],    "targets": [1, 2, 3]},     # 4 = no account: never a target
    "savings":  {"type": "ordinal", "order": [1, 2, 3, 4], "targets": [1, 2, 3, 4]},  # 5 = unknown/none: never a target
    "job":      {"type": "ordinal", "order": [1, 2, 3, 4], "targets": [1, 2, 3, 4]},
    "telephone":           {"type": "ordinal"},
    "credit_history":      {"type": "ordinal"},
    "employment_duration": {"type": "ordinal"},
    # nominal
    "purpose":                 {"type": "nominal"},
    "other_debtors":           {"type": "nominal", "targets": [1, 2, 3]},
    "property":                {"type": "nominal", "targets": [1, 2, 3]},   # 4 = unknown/none: never a target
    "other_installment_plans": {"type": "nominal", "targets": [1, 2, 3]},
    "housing":                 {"type": "nominal", "targets": [1, 2, 3]},
}

# Actionability comes from cell 5 only
for f, spec in FEATURE_SPEC.items():
    spec["action"] = "actionable" if f in actionable_features else "fixed"

MODEL_FEATURES = list(x_train_model.columns)
assert set(MODEL_FEATURES) == set(FEATURE_SPEC), "spec and model features differ"

In [ ]:
# Readable labels for displaying counterfactuals
FEATURE_MAPPINGS = {
    'status': {1: '<0 DM', 2: '0-200 DM', 3: '≥200 DM/salary', 4: 'no checking account'},
    'credit_history': {1: 'no credits/all paid', 2: 'all paid at bank', 3: 'existing paid',
                       4: 'past delays', 5: 'critical/other credits'},
    'purpose': {1: 'new car', 2: 'used car', 3: 'furniture/equipment', 4: 'radio/TV',
                5: 'domestic appliances', 6: 'repairs', 7: 'education', 8: 'retraining',
                9: 'business', 10: 'others'},
    'savings': {1: '<100 DM', 2: '100-500 DM', 3: '500-1000 DM', 4: '≥1000 DM', 5: 'unknown/none'},
    'employment_duration': {1: 'unemployed', 2: '<1 year', 3: '1-4 years', 4: '4-7 years', 5: '≥7 years'},
    'other_debtors': {1: 'none', 2: 'co-applicant', 3: 'guarantor'},
    'property': {1: 'real estate', 2: 'building society/life insurance', 3: 'car/other', 4: 'unknown/none'},
    'other_installment_plans': {1: 'bank', 2: 'stores', 3: 'none'},
    'housing': {1: 'rent', 2: 'own', 3: 'for free'},
    'job': {1: 'unemployed/unskilled non-resident', 2: 'unskilled resident',
            3: 'skilled/official', 4: 'management/self-employed/highly skilled'},
    'telephone': {1: 'none', 2: 'yes, registered'},
}

FEATURE_DESC = {
    'status': 'Status of checking account',
    'duration': 'Duration of credit in months',
    'credit_history': 'Credit history',
    'purpose': 'Purpose of credit',
    'amount': 'Credit amount in DM',
    'savings': 'Savings account',
    'employment_duration': 'Employment duration',
    'installment_rate': 'Installment rate (% of disposable income)',
    'other_debtors': 'Other debtors/guarantors',
    'present_residence': 'Years at present residence',
    'property': 'Property',
    'other_installment_plans': 'Other installment plans',
    'housing': 'Housing',
    'number_credits': 'Number of existing credits at this bank',
    'job': 'Job',
    'people_liable': 'Number of people financially dependent',
    'telephone': 'Telephone',
}

def map_value(feature, value):
    """Return the readable label for an encoded value."""
    return FEATURE_MAPPINGS.get(feature, {}).get(int(value), str(value))



In [ ]:
# Domain limits and MAD for numeric features (training data only)
stats = {
    f: {"min": x_train_model[f].min(),
        "max": x_train_model[f].max(),
        "mad": max((x_train_model[f] - x_train_model[f].median()).abs().median(), 1)}
    for f, s in FEATURE_SPEC.items() if s["type"] == "numeric"
}

# Rejected test applicants: predicted probability of bad credit >= 0.5
rejected = x_test_model[rf_pred_proba >= 0.5]
print(f"Rejected applicants in test set: {len(rejected)}")

with open("percof_setup.pkl", "wb") as f:
    pickle.dump({"feature_spec": FEATURE_SPEC, "model_features": MODEL_FEATURES,
                 "stats": stats, "rejected": rejected,
                 "feature_mappings": FEATURE_MAPPINGS, "feature_desc": FEATURE_DESC}, f)

Rejected applicants in test set: 44
